# v3: Ranking + Dynamic HGT + Selective Prediction

**Research Question:** Can a Dynamic Heterogeneous Graph Transformer with selective
prediction outperform flat baselines on S&P 500 cross-sectional stock ranking?

**Hypothesis:** Combining 4 types of dynamic graph edges (correlation, sector, news
mentions, co-occurrence) with HGT attention and a learnable selection head can produce
economically significant ranking predictions across multiple time horizons.

**Contribution Points:**
1. First GNN + Selective Prediction combination (SelectiveNet never applied to financial GNN)
2. Dynamic heterogeneous graph (4 edge types, monthly-updated correlation)
3. Systematic horizon ablation: 1d/5d/10d/21d/42d/63d
4. Calendar-driven + news-enhanced: predict all stocks daily
5. Large-scale validation: S&P 500, 502 stocks

**Predecessor:** v2 event-driven binary direction prediction -> AUC ~ 0.50 (EMH).
v3 switches to calendar-driven ranking with IC/ICIR/Sharpe evaluation.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# SETUP — imports, environment, reproducibility
# ══════════════════════════════════════════════════════════════════

import os, sys, time, random, warnings, gc
from collections import defaultdict
from itertools import combinations
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from scipy.sparse import csr_matrix
warnings.filterwarnings('ignore', category=FutureWarning)

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# --- Environment ---
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_FOLDER = '/content/drive/MyDrive/GNN\u6d4b\u8bd5'
    os.chdir(DRIVE_FOLDER)
    print(f'Colab: working dir = {DRIVE_FOLDER}')
    !pip install --quiet torch_geometric lightgbm
except ImportError:
    os.chdir('/Users/heruixi/Desktop/GNN-Testing')
    print(f'Local: working dir = {os.getcwd()}')

for d in ['data/fullscale', 'data/reference', 'plots', 'experiments']:
    os.makedirs(d, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# PARAMETERS — All tunable values in one place
# ══════════════════════════════════════════════════════════════════
PARAMS = {
    # --- Data paths ---
    'events_file':  'data/fullscale/sp500_news_events.parquet',
    'emb_file':     'data/fullscale/sp500_news_emb_finbert.npy',
    'sent_file':    'data/fullscale/sp500_news_sentiment_finbert.npy',
    'prices_file':  'data/reference/sp500_5y_prices.csv',
    'sectors_file': 'data/reference/sp500_sectors.csv',

    # --- Dynamic graph ---
    'corr_window':    126,    # trading days for rolling correlation
    'corr_threshold': 0.6,    # |Pearson r| edge inclusion
    'corr_step':      21,     # monthly re-estimation step

    # --- Prediction ---
    'horizons': [1, 5, 10, 21, 42, 63],  # forward return horizons (trading days)
    'default_horizon': 5,                 # primary horizon (MASTER default)

    # --- Time split ---
    'train_start': '2021-07-01',
    'train_end':   '2023-12-31',
    'val_end':     '2024-06-30',
    # test = everything after val_end

    # --- HGT architecture ---
    'hidden_channels': 64,
    'num_heads':       4,
    'num_hgt_layers':  2,
    'dropout':         0.3,

    # --- Training ---
    'lr':            1e-3,
    'weight_decay':  1e-4,
    'epochs':        100,
    'patience':      15,
    'grad_accum':    32,     # accumulate over N days before optimizer step

    # --- Portfolio evaluation ---
    'top_k':             30,   # long/short top-k stocks
    'transaction_cost':  15,   # bps round-trip

    # --- SelectiveNet ---
    'target_coverage': 0.2,
    'selection_lambda': 32.0,
}

print('Parameters:')
for k, v in PARAMS.items():
    print(f'  {k}: {v}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# LOAD ALL RAW DATA
# ══════════════════════════════════════════════════════════════════
t0 = time.time()

# --- Prices: (num_days, num_tickers) wide format ---
prices = pd.read_csv(PARAMS['prices_file'], index_col=0, parse_dates=True)
prices.index.name = 'Date'
print(f'Prices: {prices.shape}  ({prices.index[0].date()} -> {prices.index[-1].date()})')

# --- Sectors ---
sector_df = pd.read_csv(PARAMS['sectors_file'])
sector_col = [c for c in sector_df.columns if 'sector' in c.lower()][0]
ticker_col = [c for c in sector_df.columns if c != sector_col][0]
sector_map = dict(zip(sector_df[ticker_col], sector_df[sector_col]))
unique_sectors = sorted(set(sector_map.values()))
print(f'Sectors: {len(unique_sectors)} unique')

# --- Events + Embeddings + Sentiment ---
events = pd.read_parquet(PARAMS['events_file'])
events['date'] = pd.to_datetime(events['date'], utc=True).dt.tz_localize(None).dt.normalize()
emb_all = np.load(PARAMS['emb_file'])   # fully load for speed
sent_all = np.load(PARAMS['sent_file'])
print(f'Events: {len(events):,}  Emb: {emb_all.shape}  Sent: {sent_all.shape}')

# --- Canonical ticker list (intersection of all sources) ---
price_tickers = set(prices.columns)
event_tickers = set(events['ticker'].unique())
sector_tickers = set(sector_map.keys())
valid_tickers = sorted(price_tickers & event_tickers & sector_tickers)
ticker_to_id = {t: i for i, t in enumerate(valid_tickers)}
num_stocks = len(valid_tickers)
print(f'Valid tickers: {num_stocks}')

# Filter prices to valid tickers
prices = prices[valid_tickers]
returns = prices.pct_change()
returns.iloc[0] = 0

# Trading day index
all_dates = prices.index
num_days = len(all_dates)
date_to_idx = {d: i for i, d in enumerate(all_dates)}
print(f'Trading days: {num_days}  ({all_dates[0].date()} -> {all_dates[-1].date()})')

print(f'\nAll data loaded in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N1a: PRICE FEATURES — 9-dim per stock-day (strictly T-1 close)
# 3 windows (5/10/21d) x 3 stats (return mean, volatility, momentum)
# ══════════════════════════════════════════════════════════════════
t0 = time.time()

price_feat_dict = {}
for w in [5, 10, 21]:
    # Mean daily return over past w days (ending T-1)
    price_feat_dict[f'ret_mean_{w}d'] = returns.rolling(w).mean().shift(1)
    # Volatility: std of daily returns (ending T-1)
    price_feat_dict[f'ret_std_{w}d'] = returns.rolling(w).std().shift(1)
    # Momentum: cumulative return = close[T-1] / close[T-1-w] - 1
    price_feat_dict[f'momentum_{w}d'] = (prices.shift(1) / prices.shift(1 + w) - 1)

price_feat_names = list(price_feat_dict.keys())

# Stack: (num_days, num_stocks, 9)
price_features = np.stack(
    [price_feat_dict[name].values for name in price_feat_names], axis=-1
).astype(np.float32)

nan_before = np.isnan(price_features).sum()
price_features = np.nan_to_num(price_features, 0.0)
print(f'Price features: {price_features.shape}')
print(f'  Features: {price_feat_names}')
print(f'  NaN filled: {nan_before:,} -> 0 (first ~21 days lack lookback)')

del price_feat_dict
gc.collect()
print(f'N1a complete in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N1b: NEWS FEATURES -> CALENDAR GRID
# For each (trading_day, ticker): mean-pooled FinBERT 768d + 3d sentiment
# Days without news -> zero vector + has_news=0
# Also precompute per-day event metadata for graph construction (N2)
# ══════════════════════════════════════════════════════════════════
t0 = time.time()

# Map events to calendar grid
events['event_idx'] = np.arange(len(events))
events_v = events[events['ticker'].isin(ticker_to_id)].copy()
events_v['stock_id'] = events_v['ticker'].map(ticker_to_id)
events_v['day_idx'] = events_v['date'].map(date_to_idx)
events_v = events_v.dropna(subset=['day_idx'])
events_v['day_idx'] = events_v['day_idx'].astype(int)
print(f'Events mapped to grid: {len(events_v):,} / {len(events):,}')

# --- Mean-pool via sparse aggregation (fast) ---
group_keys = events_v['day_idx'].values * num_stocks + events_v['stock_id'].values
unique_keys, inverse = np.unique(group_keys, return_inverse=True)
group_counts = np.bincount(inverse).astype(np.float32)

# Sparse aggregation matrix: (num_groups, num_events_valid)
rows = inverse
cols = np.arange(len(events_v))
vals = 1.0 / group_counts[inverse]
agg_mat = csr_matrix((vals, (rows, cols)), shape=(len(unique_keys), len(events_v)))

# Aggregate embeddings and sentiment
event_idxs = events_v['event_idx'].values
agg_emb = (agg_mat @ emb_all[event_idxs]).astype(np.float32)   # (groups, 768)
agg_sent = (agg_mat @ sent_all[event_idxs]).astype(np.float32)  # (groups, 3)

# Scatter into (num_days, num_stocks, dim) arrays
news_emb = np.zeros((num_days, num_stocks, 768), dtype=np.float32)
news_sent = np.zeros((num_days, num_stocks, 3), dtype=np.float32)
has_news = np.zeros((num_days, num_stocks, 1), dtype=np.float32)

for i, gk in enumerate(unique_keys):
    d_idx = int(gk // num_stocks)
    s_idx = int(gk % num_stocks)
    news_emb[d_idx, s_idx] = agg_emb[i]
    news_sent[d_idx, s_idx] = agg_sent[i]
    has_news[d_idx, s_idx] = 1.0

news_coverage = has_news.sum() / (num_days * num_stocks) * 100
print(f'News coverage: {has_news.sum():.0f} / {num_days * num_stocks:,} stock-days '
      f'({news_coverage:.1f}%)')

# --- Combine all features: (num_days, num_stocks, 781) ---
# Layout: [price(9) | emb(768) | sent(3) | has_news(1)]
stock_features_np = np.concatenate(
    [price_features, news_emb, news_sent, has_news], axis=-1
)
print(f'Stock features: {stock_features_np.shape}  '
      f'({stock_features_np.nbytes / 1e9:.2f} GB)')

# --- Precompute per-day event data for graph construction (N2) ---
# Store only indices to save memory; load embeddings lazily
daily_events = {}
events_v_sorted = events_v.sort_values(['day_idx', 'stock_id'])
for day_idx, group in events_v_sorted.groupby('day_idx'):
    daily_events[int(day_idx)] = {
        'event_idxs': group['event_idx'].values,
        'stock_ids': group['stock_id'].values,
        'titles': group['title'].values if 'title' in group.columns else None,
    }

print(f'Daily event groups: {len(daily_events)} days')

del price_features, news_emb, news_sent, has_news, agg_emb, agg_sent, agg_mat
gc.collect()
print(f'\nN1b complete in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N1c: MULTI-HORIZON LABELS (6 horizons)
# Label = Z-score of cross-sectional excess return
# Excess = stock_ret - equal_weight_market_ret
# Z-score = per-day normalization across stocks
# ══════════════════════════════════════════════════════════════════
t0 = time.time()

labels = {}  # horizon -> (num_days, num_stocks) np array
label_valid = {}  # horizon -> (num_days, num_stocks) bool mask

for h in PARAMS['horizons']:
    # Forward return: (close[T+h] - close[T]) / close[T]
    fwd_ret = prices.shift(-h) / prices - 1  # (num_days, num_stocks)

    # Equal-weight market return per day
    market_ret = fwd_ret.mean(axis=1)  # (num_days,)

    # Excess return
    excess = fwd_ret.sub(market_ret, axis=0)  # (num_days, num_stocks)

    # Cross-sectional Z-score per day
    day_mean = excess.mean(axis=1)
    day_std = excess.std(axis=1)
    day_std[day_std < 1e-8] = 1.0  # avoid division by zero
    z_score = excess.sub(day_mean, axis=0).div(day_std, axis=0)

    # Valid mask: not NaN (last h days have NaN forward returns)
    valid = ~z_score.isna()

    labels[h] = z_score.values.astype(np.float32)
    labels[h] = np.nan_to_num(labels[h], 0.0)
    label_valid[h] = valid.values

    n_valid = valid.sum().sum()
    print(f'Horizon {h:2d}d: {n_valid:,} valid labels '
          f'(last {h} days = NaN, z-score mean={labels[h][valid.values].mean():.4f} '
          f'std={labels[h][valid.values].std():.4f})')

print(f'\nN1c complete in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N1d: TIME SPLIT + DATA SUMMARY
# Train: 2021-07 -> 2023-12 | Val: 2024-01 -> 2024-06 | Test: 2024-07+
# ══════════════════════════════════════════════════════════════════

train_start = pd.Timestamp(PARAMS['train_start'])
train_end = pd.Timestamp(PARAMS['train_end'])
val_end = pd.Timestamp(PARAMS['val_end'])

train_mask = (all_dates >= train_start) & (all_dates <= train_end)
val_mask = (all_dates > train_end) & (all_dates <= val_end)
test_mask = (all_dates > val_end)

train_days = np.where(train_mask)[0]
val_days = np.where(val_mask)[0]
test_days = np.where(test_mask)[0]

print(f'=== Time Split ===')
print(f'Train: {all_dates[train_days[0]].date()} -> {all_dates[train_days[-1]].date()} '
      f'({len(train_days)} days)')
print(f'Val:   {all_dates[val_days[0]].date()} -> {all_dates[val_days[-1]].date()} '
      f'({len(val_days)} days)')
print(f'Test:  {all_dates[test_days[0]].date()} -> {all_dates[test_days[-1]].date()} '
      f'({len(test_days)} days)')

# Summary: stock-days with news per split
for name, days in [('Train', train_days), ('Val', val_days), ('Test', test_days)]:
    total = len(days) * num_stocks
    with_news = stock_features_np[days, :, -1].sum()  # has_news flag is last dim
    print(f'  {name}: {total:,} stock-days, {int(with_news):,} with news '
          f'({with_news/total*100:.1f}%)')

# Convert features to tensors
stock_features_t = torch.tensor(stock_features_np, dtype=torch.float32)
labels_t = {h: torch.tensor(labels[h], dtype=torch.float32) for h in PARAMS['horizons']}
label_valid_t = {h: torch.tensor(label_valid[h], dtype=torch.bool) for h in PARAMS['horizons']}

print(f'\nFeature tensor: {stock_features_t.shape}  ({stock_features_t.element_size() * stock_features_t.nelement() / 1e9:.2f} GB)')
print(f'Label tensors: {len(labels_t)} horizons')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N2: GRAPH CONSTRUCTION — 4 edge types
# 1. stock-stock (correlation): monthly dynamic, |r|>0.6, w=126
# 2. stock-stock (sector): static, same GICS sector
# 3. news->stock (mentions): daily, per event
# 4. stock-stock (co-occurrence): daily, same-article tickers
# ══════════════════════════════════════════════════════════════════
t0 = time.time()

# ------------------------------------------------------------------
# Edge Type 1: Correlation (monthly dynamic)
# ------------------------------------------------------------------
corr_w = PARAMS['corr_window']
corr_t = PARAMS['corr_threshold']
corr_step = PARAMS['corr_step']

# Build correlation edge snapshots
corr_snapshots = {}  # snapshot_idx -> edge_index tensor
corr_day_to_snapshot = {}  # day_idx -> snapshot_idx

snapshot_points = list(range(corr_w, num_days, corr_step))
print(f'Correlation snapshots: {len(snapshot_points)} (w={corr_w}, t={corr_t}, step={corr_step})')

for snap_idx, t_end in enumerate(snapshot_points):
    window_ret = returns.iloc[t_end - corr_w : t_end].values  # (w, num_stocks)
    corr_mat = np.corrcoef(window_ret.T)  # (num_stocks, num_stocks)
    np.fill_diagonal(corr_mat, 0)
    mask = np.abs(corr_mat) > corr_t
    src, dst = np.where(mask)
    corr_snapshots[snap_idx] = torch.tensor(
        np.stack([src, dst]), dtype=torch.long
    )

# Map each day to its most recent correlation snapshot
snap_idx = 0
for day_idx in range(num_days):
    # Find latest snapshot that ends at or before this day
    while snap_idx + 1 < len(snapshot_points) and snapshot_points[snap_idx + 1] <= day_idx:
        snap_idx += 1
    if snapshot_points[snap_idx] <= day_idx:
        corr_day_to_snapshot[day_idx] = snap_idx
    # Days before first snapshot use snapshot 0
    elif snap_idx == 0:
        corr_day_to_snapshot[day_idx] = 0

# Stats
for i in [0, len(snapshot_points)//2, len(snapshot_points)-1]:
    e = corr_snapshots[i]
    print(f'  Snapshot {i}: {e.shape[1]} edges, '
          f'density={e.shape[1]/(num_stocks*(num_stocks-1))*100:.1f}%')

# ------------------------------------------------------------------
# Edge Type 2: Sector (static)
# ------------------------------------------------------------------
sector_edges_src, sector_edges_dst = [], []
sector_groups = defaultdict(list)
for t_name in valid_tickers:
    if t_name in sector_map:
        sector_groups[sector_map[t_name]].append(ticker_to_id[t_name])

for sector, members in sector_groups.items():
    for i in range(len(members)):
        for j in range(len(members)):
            if i != j:
                sector_edges_src.append(members[i])
                sector_edges_dst.append(members[j])

sector_edge_index = torch.tensor(
    [sector_edges_src, sector_edges_dst], dtype=torch.long
)
print(f'Sector edges: {sector_edge_index.shape[1]:,} '
      f'({len(sector_groups)} sectors)')

# ------------------------------------------------------------------
# Edge Types 3 & 4: News mention + co-occurrence (daily)
# ------------------------------------------------------------------
daily_graph_data = {}  # day_idx -> dict with mention_edges, cooccur_edges, news_feat

for day_idx in range(num_days):
    if day_idx not in daily_events:
        daily_graph_data[day_idx] = {
            'n_news': 0,
            'news_feat': torch.zeros(1, 771, dtype=torch.float32),  # dummy node
            'mention_edges': torch.zeros(2, 0, dtype=torch.long),
            'cooccur_edges': torch.zeros(2, 0, dtype=torch.long),
        }
        continue

    de = daily_events[day_idx]
    ev_idxs = de['event_idxs']
    stock_ids = de['stock_ids']
    n_events = len(ev_idxs)

    # News node features: FinBERT embedding + sentiment = 771-dim
    feat = np.concatenate([emb_all[ev_idxs], sent_all[ev_idxs]], axis=1)  # (N, 771)

    # Mention edges: news_node_i -> stock_id
    mention_edges = torch.tensor(
        np.stack([np.arange(n_events), stock_ids]), dtype=torch.long
    )

    # Co-occurrence: stocks cited in same article (group by title)
    cooccur_src, cooccur_dst = [], []
    if de['titles'] is not None:
        title_to_stocks = defaultdict(set)
        for title, sid in zip(de['titles'], stock_ids):
            title_to_stocks[title].add(int(sid))
        for title, sids in title_to_stocks.items():
            sids = list(sids)
            if len(sids) >= 2:
                for s1, s2 in combinations(sids, 2):
                    cooccur_src.extend([s1, s2])
                    cooccur_dst.extend([s2, s1])

    if cooccur_src:
        cooccur_edges = torch.tensor([cooccur_src, cooccur_dst], dtype=torch.long)
    else:
        cooccur_edges = torch.zeros(2, 0, dtype=torch.long)

    daily_graph_data[day_idx] = {
        'n_news': n_events,
        'news_feat': torch.tensor(feat, dtype=torch.float32),
        'mention_edges': mention_edges,
        'cooccur_edges': cooccur_edges,
    }

# Stats
n_mention_total = sum(d['mention_edges'].shape[1] for d in daily_graph_data.values())
n_cooccur_total = sum(d['cooccur_edges'].shape[1] for d in daily_graph_data.values())
print(f'News mention edges: {n_mention_total:,} total '
      f'({n_mention_total/num_days:.0f}/day avg)')
print(f'Co-occurrence edges: {n_cooccur_total:,} total '
      f'({n_cooccur_total/num_days:.0f}/day avg)')

# ------------------------------------------------------------------
# HeteroData metadata
# ------------------------------------------------------------------
METADATA = (
    ['stock', 'news'],
    [
        ('stock', 'corr', 'stock'),
        ('stock', 'sector', 'stock'),
        ('news', 'mentions', 'stock'),
        ('stock', 'cooccur', 'stock'),
    ]
)
# ------------------------------------------------------------------
# N2d: Jaccard Audit — correlation edge dynamics across snapshots
# ------------------------------------------------------------------
jaccard_values = []
jaccard_dates = []
for i in range(1, len(snapshot_points)):
    s1 = set(map(tuple, corr_snapshots[i-1].t().numpy().tolist()))
    s2 = set(map(tuple, corr_snapshots[i].t().numpy().tolist()))
    union = len(s1 | s2)
    jaccard = len(s1 & s2) / union if union > 0 else 1.0
    jaccard_values.append(jaccard)
    jaccard_dates.append(all_dates[snapshot_points[i]])

print(f'\nJaccard similarity (adjacent snapshots):')
print(f'  Mean: {np.mean(jaccard_values):.3f}  Std: {np.std(jaccard_values):.3f}')
print(f'  Min:  {np.min(jaccard_values):.3f} at {jaccard_dates[np.argmin(jaccard_values)].date()}')

print(f'\nGraph metadata:')
print(f'  Node types: {METADATA[0]}')
print(f'  Edge types: {[e[1] for e in METADATA[1]]}')
print(f'\nN2 complete in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# HELPER: Build HeteroData for a single day
# ══════════════════════════════════════════════════════════════════
from torch_geometric.data import HeteroData

def build_hetero_data(day_idx, to_device=None):
    """Construct HeteroData graph for a single trading day."""
    dgd = daily_graph_data[day_idx]

    data = HeteroData()

    # --- Stock node features ---
    data['stock'].x = stock_features_t[day_idx]  # (num_stocks, 781)
    data['stock'].num_nodes = num_stocks

    # --- News node features ---
    if dgd['n_news'] > 0:
        data['news'].x = dgd['news_feat']
        data['news'].num_nodes = dgd['n_news']
    else:
        # Dummy news node (ensures metadata consistency)
        data['news'].x = dgd['news_feat']  # (1, 771) zeros
        data['news'].num_nodes = 1

    # --- Edge type 1: Correlation ---
    snap_idx = corr_day_to_snapshot.get(day_idx, 0)
    data['stock', 'corr', 'stock'].edge_index = corr_snapshots[snap_idx]

    # --- Edge type 2: Sector (static) ---
    data['stock', 'sector', 'stock'].edge_index = sector_edge_index

    # --- Edge type 3: News mentions ---
    if dgd['n_news'] > 0:
        data['news', 'mentions', 'stock'].edge_index = dgd['mention_edges']
    else:
        data['news', 'mentions', 'stock'].edge_index = torch.zeros(2, 0, dtype=torch.long)

    # --- Edge type 4: Co-occurrence ---
    data['stock', 'cooccur', 'stock'].edge_index = dgd['cooccur_edges']

    if to_device is not None:
        data = data.to(to_device)
    return data

# Validation: build sample graph
sample_data = build_hetero_data(train_days[len(train_days)//2])
print('=== Sample HeteroData ===')
print(sample_data)
for et in METADATA[1]:
    ei = sample_data[et].edge_index
    print(f'  {et[1]}: {ei.shape[1]} edges')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# EVALUATION UTILITIES — IC, ICIR, Portfolio Backtest
# ══════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

def compute_daily_ic(predictions, actuals, valid_mask, day_indices):
    """
    Compute daily IC (Spearman correlation) cross-sectionally.

    Parameters
    ----------
    predictions : np.ndarray, shape (num_all_days, num_stocks)
    actuals : np.ndarray, shape (num_all_days, num_stocks)
    valid_mask : np.ndarray, shape (num_all_days, num_stocks)
    day_indices : array of day indices to evaluate

    Returns
    -------
    ic_series : np.ndarray of daily IC values
    dates : corresponding dates
    """
    ic_list = []
    valid_dates = []
    for d in day_indices:
        mask = valid_mask[d]
        pred = predictions[d][mask]
        actual = actuals[d][mask]
        if len(pred) >= 30:
            ic, _ = spearmanr(pred, actual)
            if not np.isnan(ic):
                ic_list.append(ic)
                valid_dates.append(all_dates[d])
    return np.array(ic_list), valid_dates


def compute_metrics(predictions, horizon, day_indices, name=''):
    """Compute full ranking metrics for a model's predictions."""
    actuals = labels[horizon]
    valid = label_valid[horizon]

    ic_arr, ic_dates = compute_daily_ic(predictions, actuals, valid, day_indices)

    if len(ic_arr) == 0:
        return {'name': name, 'IC': 0, 'ICIR': 0, 'Rank_IC': 0}

    ic_mean = ic_arr.mean()
    ic_std = ic_arr.std()
    icir = ic_mean / ic_std if ic_std > 1e-8 else 0

    # --- Portfolio backtest ---
    top_k = PARAMS['top_k']
    tc_bps = PARAMS['transaction_cost']
    daily_long_ret = []
    daily_short_ret = []
    daily_ls_ret = []

    fwd_raw = prices.pct_change(horizon).shift(-horizon).values  # raw returns

    for d in day_indices:
        mask = valid[d]
        if mask.sum() < 2 * top_k:
            continue
        pred = predictions[d].copy()
        pred[~mask] = np.nan

        # Rank stocks
        valid_idx = np.where(mask)[0]
        scores = pred[valid_idx]
        ranks = np.argsort(np.argsort(-scores))  # 0 = best

        top_idx = valid_idx[ranks < top_k]
        bot_idx = valid_idx[ranks >= len(scores) - top_k]

        # Raw returns for this horizon
        ret_day = fwd_raw[d]
        long_ret = np.nanmean(ret_day[top_idx])
        short_ret = np.nanmean(ret_day[bot_idx])
        ls_ret = long_ret - short_ret

        daily_long_ret.append(long_ret)
        daily_short_ret.append(short_ret)
        daily_ls_ret.append(ls_ret)

    daily_ls_ret = np.array(daily_ls_ret)
    daily_long_ret = np.array(daily_long_ret)

    # Annualize (scale by 252/horizon for daily-equivalent)
    scale = 252 / horizon
    ann_ls = daily_ls_ret.mean() * scale
    ann_ls_vol = daily_ls_ret.std() * np.sqrt(scale) if len(daily_ls_ret) > 1 else 1e-8
    sharpe_ls = ann_ls / ann_ls_vol if ann_ls_vol > 1e-8 else 0

    ann_long = daily_long_ret.mean() * scale
    ann_long_vol = daily_long_ret.std() * np.sqrt(scale) if len(daily_long_ret) > 1 else 1e-8
    sharpe_long = ann_long / ann_long_vol if ann_long_vol > 1e-8 else 0

    # Transaction costs (rebalance each period)
    n_rebal = len(daily_ls_ret)
    tc_total = n_rebal * (tc_bps / 10000) * 2  # 2x for long+short turnover (approx)
    ann_ls_net = ann_ls - tc_total / max(1, n_rebal / scale)

    # Max drawdown
    cum = np.cumsum(daily_ls_ret)
    peak = np.maximum.accumulate(cum)
    max_dd = (peak - cum).max() if len(cum) > 0 else 0

    return {
        'name': name,
        'IC': round(ic_mean, 5),
        'ICIR': round(icir, 3),
        'IC_std': round(ic_std, 5),
        'Sharpe_LS': round(sharpe_ls, 3),
        'Sharpe_Long': round(sharpe_long, 3),
        'Ann_LS': f'{ann_ls*100:.2f}%',
        'Ann_LS_net': f'{ann_ls_net*100:.2f}%',
        'MaxDD': f'{max_dd*100:.2f}%',
        'n_days': len(ic_arr),
    }


def print_results_table(results_list, title=''):
    """Pretty-print a list of metric dicts."""
    if title:
        print(f'\n=== {title} ===')
    df = pd.DataFrame(results_list)
    display_cols = ['name', 'IC', 'ICIR', 'Sharpe_LS', 'Sharpe_Long',
                    'Ann_LS', 'Ann_LS_net', 'MaxDD', 'n_days']
    cols = [c for c in display_cols if c in df.columns]
    print(df[cols].to_string(index=False))
    return df

print('Evaluation utilities loaded.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N3a: NON-GNN BASELINES (LR, XGBoost, LightGBM)
# Predict ranking scores without graph structure
# ══════════════════════════════════════════════════════════════════
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print('XGBoost not available, skipping')
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print('LightGBM not available, skipping')

t0 = time.time()
h = PARAMS['default_horizon']
print(f'Running baselines for {h}d horizon...')

# Flatten features for sklearn: (days * stocks, features)
def flatten_data(day_indices, features_np, labels_np, valid_np):
    """Flatten (day, stock) grid into (sample, features) for sklearn."""
    X_list, y_list = [], []
    for d in day_indices:
        mask = valid_np[d]
        X_list.append(features_np[d][mask])
        y_list.append(labels_np[d][mask])
    return np.vstack(X_list), np.concatenate(y_list)

# Prepare data
X_train, y_train = flatten_data(train_days, stock_features_np, labels[h], label_valid[h])
X_val, y_val = flatten_data(val_days, stock_features_np, labels[h], label_valid[h])
X_test, y_test = flatten_data(test_days, stock_features_np, labels[h], label_valid[h])
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

# Standardize
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

# Feature subsets for ablation
PRICE_DIM = 9
NEWS_DIM = 768 + 3 + 1  # emb + sent + has_news = 772

baseline_results = []

# --- B1: Ridge (price features only, 9-dim) ---
ridge_price = Ridge(alpha=1.0)
ridge_price.fit(X_train_s[:, :PRICE_DIM], y_train)

# Predict per-day for proper IC evaluation
pred_price = np.zeros((num_days, num_stocks), dtype=np.float32)
for d in range(num_days):
    x = scaler.transform(stock_features_np[d])[:, :PRICE_DIM]
    pred_price[d] = ridge_price.predict(x)

res = compute_metrics(pred_price, h, test_days, 'B1: Ridge (price 9d)')
baseline_results.append(res)
print(f'B1 done: IC={res["IC"]:.5f}')

# --- B2: Ridge (price + news, 781-dim) ---
ridge_all = Ridge(alpha=1.0)
ridge_all.fit(X_train_s, y_train)

pred_all = np.zeros((num_days, num_stocks), dtype=np.float32)
for d in range(num_days):
    x = scaler.transform(stock_features_np[d])
    pred_all[d] = ridge_all.predict(x)

res = compute_metrics(pred_all, h, test_days, 'B2: Ridge (all 781d)')
baseline_results.append(res)
print(f'B2 done: IC={res["IC"]:.5f}')

# --- B3: XGBoost ---
if HAS_XGB:
    xgb_model = xgb.XGBRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        early_stopping_rounds=20, eval_metric='rmse',
        random_state=SEED, n_jobs=-1, verbosity=0,
    )
    xgb_model.fit(X_train_s, y_train, eval_set=[(X_val_s, y_val)], verbose=False)

    pred_xgb = np.zeros((num_days, num_stocks), dtype=np.float32)
    for d in range(num_days):
        x = scaler.transform(stock_features_np[d])
        pred_xgb[d] = xgb_model.predict(x)

    res = compute_metrics(pred_xgb, h, test_days, 'B3: XGBoost')
    baseline_results.append(res)
    print(f'B3 done: IC={res["IC"]:.5f}')

# --- B4: LightGBM ---
if HAS_LGB:
    lgb_model = lgb.LGBMRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=-1, verbose=-1,
    )
    lgb_model.fit(
        X_train_s, y_train,
        eval_set=[(X_val_s, y_val)],
        callbacks=[lgb.early_stopping(20, verbose=False)],
    )

    pred_lgb = np.zeros((num_days, num_stocks), dtype=np.float32)
    for d in range(num_days):
        x = scaler.transform(stock_features_np[d])
        pred_lgb[d] = lgb_model.predict(x)

    res = compute_metrics(pred_lgb, h, test_days, 'B4: LightGBM')
    baseline_results.append(res)
    print(f'B4 done: IC={res["IC"]:.5f}')

print_results_table(baseline_results, f'Non-GNN Baselines ({h}d horizon, test set)')
print(f'\nBaselines complete in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N3b: HGT MODEL DEFINITION
# Heterogeneous Graph Transformer for cross-sectional ranking
# ══════════════════════════════════════════════════════════════════
from torch_geometric.nn import HGTConv

class RankingHGT(nn.Module):
    """
    Heterogeneous Graph Transformer for stock ranking prediction.

    Node types: stock (781-dim), news (771-dim)
    Edge types: corr, sector, mentions, cooccur
    Output: per-stock ranking score (scalar)
    """
    def __init__(self, stock_in, news_in, hidden, metadata,
                 num_heads=4, num_layers=2, dropout=0.3):
        super().__init__()
        self.stock_lin = nn.Linear(stock_in, hidden)
        self.news_lin = nn.Linear(news_in, hidden)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(HGTConv(hidden, hidden, metadata, heads=num_heads))
            self.norms.append(nn.LayerNorm(hidden))

        self.dropout = dropout
        self.ranking_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x_dict, edge_index_dict):
        # Project to hidden space
        h_dict = {
            'stock': F.relu(self.stock_lin(x_dict['stock'])),
        }
        if 'news' in x_dict and x_dict['news'].shape[0] > 0:
            h_dict['news'] = F.relu(self.news_lin(x_dict['news']))
        else:
            h_dict['news'] = torch.zeros(1, self.stock_lin.out_features,
                                          device=x_dict['stock'].device)

        # HGT message passing with residual + LayerNorm
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h_dict, edge_index_dict)
            # Residual + norm for stock nodes
            h_dict['stock'] = norm(
                F.dropout(h_new['stock'], p=self.dropout, training=self.training)
                + h_dict['stock']
            )
            # News nodes: keep original (no incoming edges update them)
            if 'news' in h_new:
                h_dict['news'] = h_new['news']

        # Ranking score for each stock
        return self.ranking_head(h_dict['stock']).squeeze(-1)

    def get_stock_embeddings(self, x_dict, edge_index_dict):
        """Return stock node embeddings before ranking head (for SelectiveNet)."""
        h_dict = {
            'stock': F.relu(self.stock_lin(x_dict['stock'])),
        }
        if 'news' in x_dict and x_dict['news'].shape[0] > 0:
            h_dict['news'] = F.relu(self.news_lin(x_dict['news']))
        else:
            h_dict['news'] = torch.zeros(1, self.stock_lin.out_features,
                                          device=x_dict['stock'].device)
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h_dict, edge_index_dict)
            h_dict['stock'] = norm(
                F.dropout(h_new['stock'], p=self.dropout, training=self.training)
                + h_dict['stock']
            )
            if 'news' in h_new:
                h_dict['news'] = h_new['news']
        return h_dict['stock']


# --- Also define simpler GNN baselines for ablation ---

class RankingGNN(nn.Module):
    """Homogeneous GNN (GraphSAGE or GAT) for ablation against HGT.
    Uses only stock nodes and correlation+sector edges."""
    def __init__(self, in_channels, hidden, conv_type='sage', num_layers=2, dropout=0.3):
        super().__init__()
        self.lin = nn.Linear(in_channels, hidden)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(num_layers):
            if conv_type == 'sage':
                from torch_geometric.nn import SAGEConv
                self.convs.append(SAGEConv(hidden, hidden))
            elif conv_type == 'gat':
                from torch_geometric.nn import GATConv
                self.convs.append(GATConv(hidden, hidden // 4, heads=4, concat=True))
            self.norms.append(nn.LayerNorm(hidden))

        self.dropout = dropout
        self.ranking_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x, edge_index):
        h = F.relu(self.lin(x))
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index)
            h = norm(F.dropout(h_new, p=self.dropout, training=self.training) + h)
        return self.ranking_head(h).squeeze(-1)

    def get_stock_embeddings(self, x, edge_index):
        """Return stock node embeddings before ranking head (for SelectiveNet)."""
        h = F.relu(self.lin(x))
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index)
            h = norm(F.dropout(h_new, p=self.dropout, training=self.training) + h)
        return h


print('Models defined: RankingHGT, RankingGNN')
print(f'  HGT: stock_in=781, news_in=771, hidden={PARAMS["hidden_channels"]}, '
      f'heads={PARAMS["num_heads"]}, layers={PARAMS["num_hgt_layers"]}')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N3c: GNN TRAINING LOOP
# Per-day graph processing with gradient accumulation
# ══════════════════════════════════════════════════════════════════

def train_hgt(model, horizon, train_days, val_days, epochs=None, patience=None,
              grad_accum=None, verbose=True):
    """
    Train HGT model for ranking prediction on a specific horizon.

    Returns: best model state dict, training history
    """
    epochs = epochs or PARAMS['epochs']
    patience = patience or PARAMS['patience']
    grad_accum = grad_accum or PARAMS['grad_accum']

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(),
                                 lr=PARAMS['lr'],
                                 weight_decay=PARAMS['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-5
    )

    best_val_loss = float('inf')
    best_state = None
    no_improve = 0
    history = {'train_loss': [], 'val_loss': [], 'val_ic': []}

    lab = labels_t[horizon]
    valid = label_valid_t[horizon]

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        train_loss_sum = 0
        train_count = 0
        optimizer.zero_grad()

        # Shuffle training days
        day_order = train_days[np.random.permutation(len(train_days))]

        for step, d in enumerate(day_order):
            data = build_hetero_data(d, to_device=device)
            pred = model(data.x_dict, data.edge_index_dict)

            # Loss on valid labels only
            mask = valid[d].to(device)
            target = lab[d].to(device)

            if mask.sum() < 10:
                continue

            loss = F.mse_loss(pred[mask], target[mask])
            loss = loss / grad_accum
            loss.backward()
            train_loss_sum += loss.item() * grad_accum
            train_count += 1

            if (step + 1) % grad_accum == 0 or step == len(day_order) - 1:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()

        avg_train = train_loss_sum / max(train_count, 1)
        history['train_loss'].append(avg_train)

        # --- Validate ---
        model.eval()
        val_loss_sum = 0
        val_count = 0
        val_preds = np.zeros((num_days, num_stocks), dtype=np.float32)

        with torch.no_grad():
            for d in val_days:
                data = build_hetero_data(d, to_device=device)
                pred = model(data.x_dict, data.edge_index_dict)
                mask = valid[d].to(device)
                target = lab[d].to(device)
                if mask.sum() < 10:
                    continue
                loss = F.mse_loss(pred[mask], target[mask])
                val_loss_sum += loss.item()
                val_count += 1
                val_preds[d] = pred.cpu().numpy()

        avg_val = val_loss_sum / max(val_count, 1)
        history['val_loss'].append(avg_val)

        # Val IC
        ic_arr, _ = compute_daily_ic(val_preds, labels[horizon],
                                      label_valid[horizon], val_days)
        val_ic = ic_arr.mean() if len(ic_arr) > 0 else 0
        history['val_ic'].append(val_ic)

        scheduler.step(avg_val)

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if verbose and (epoch + 1) % 5 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(f'  Epoch {epoch+1:3d}: train_loss={avg_train:.5f} '
                  f'val_loss={avg_val:.5f} val_IC={val_ic:.5f} lr={lr:.1e}')

        if no_improve >= patience:
            if verbose:
                print(f'  Early stop at epoch {epoch+1}')
            break

    # Load best
    model.load_state_dict(best_state)
    model = model.to(device)
    return model, history


def evaluate_hgt(model, horizon, day_indices, name=''):
    """Evaluate trained HGT model on given day indices."""
    model.eval()
    preds = np.zeros((num_days, num_stocks), dtype=np.float32)

    with torch.no_grad():
        for d in day_indices:
            data = build_hetero_data(d, to_device=device)
            pred = model(data.x_dict, data.edge_index_dict)
            preds[d] = pred.cpu().numpy()

    return compute_metrics(preds, horizon, day_indices, name)


def train_homogeneous_gnn(model, horizon, train_days, val_days,
                           edge_types=['corr', 'sector'],
                           epochs=None, patience=None, grad_accum=None, verbose=True):
    """Train a homogeneous GNN using only stock-stock edges."""
    epochs = epochs or PARAMS['epochs']
    patience = patience or PARAMS['patience']
    grad_accum = grad_accum or PARAMS['grad_accum']

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=PARAMS['lr'],
                                 weight_decay=PARAMS['weight_decay'])

    best_val_loss = float('inf')
    best_state = None
    no_improve = 0

    lab = labels_t[horizon]
    valid = label_valid_t[horizon]

    for epoch in range(epochs):
        model.train()
        train_loss_sum = 0
        train_count = 0
        optimizer.zero_grad()

        day_order = train_days[np.random.permutation(len(train_days))]
        for step, d in enumerate(day_order):
            x = stock_features_t[d].to(device)

            # Combine requested edge types
            edge_lists = []
            if 'corr' in edge_types:
                snap = corr_day_to_snapshot.get(d, 0)
                edge_lists.append(corr_snapshots[snap])
            if 'sector' in edge_types:
                edge_lists.append(sector_edge_index)
            if 'cooccur' in edge_types:
                edge_lists.append(daily_graph_data[d]['cooccur_edges'])
            edge_index = torch.cat(edge_lists, dim=1).to(device) if edge_lists else torch.zeros(2,0,dtype=torch.long,device=device)

            pred = model(x, edge_index)
            mask = valid[d].to(device)
            target = lab[d].to(device)
            if mask.sum() < 10:
                continue
            loss = F.mse_loss(pred[mask], target[mask]) / grad_accum
            loss.backward()
            train_loss_sum += loss.item() * grad_accum
            train_count += 1
            if (step + 1) % grad_accum == 0 or step == len(day_order) - 1:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()

        avg_train = train_loss_sum / max(train_count, 1)

        # Validate
        model.eval()
        val_loss_sum = 0
        val_count = 0
        with torch.no_grad():
            for d in val_days:
                x = stock_features_t[d].to(device)
                edge_lists = []
                if 'corr' in edge_types:
                    snap = corr_day_to_snapshot.get(d, 0)
                    edge_lists.append(corr_snapshots[snap])
                if 'sector' in edge_types:
                    edge_lists.append(sector_edge_index)
                if 'cooccur' in edge_types:
                    edge_lists.append(daily_graph_data[d]['cooccur_edges'])
                edge_index = torch.cat(edge_lists, dim=1).to(device) if edge_lists else torch.zeros(2,0,dtype=torch.long,device=device)
                pred = model(x, edge_index)
                mask = valid[d].to(device)
                target = lab[d].to(device)
                if mask.sum() < 10:
                    continue
                loss = F.mse_loss(pred[mask], target[mask])
                val_loss_sum += loss.item()
                val_count += 1

        avg_val = val_loss_sum / max(val_count, 1)
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
        if verbose and (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d}: train={avg_train:.5f} val={avg_val:.5f}')
        if no_improve >= patience:
            if verbose:
                print(f'  Early stop at epoch {epoch+1}')
            break

    model.load_state_dict(best_state)
    model = model.to(device)
    return model


def evaluate_homogeneous_gnn(model, horizon, day_indices, edge_types=['corr','sector'], name=''):
    """Evaluate homogeneous GNN."""
    model.eval()
    preds = np.zeros((num_days, num_stocks), dtype=np.float32)
    with torch.no_grad():
        for d in day_indices:
            x = stock_features_t[d].to(device)
            edge_lists = []
            if 'corr' in edge_types:
                snap = corr_day_to_snapshot.get(d, 0)
                edge_lists.append(corr_snapshots[snap])
            if 'sector' in edge_types:
                edge_lists.append(sector_edge_index)
            if 'cooccur' in edge_types:
                edge_lists.append(daily_graph_data[d]['cooccur_edges'])
            edge_index = torch.cat(edge_lists, dim=1).to(device) if edge_lists else torch.zeros(2,0,dtype=torch.long,device=device)
            pred = model(x, edge_index)
            preds[d] = pred.cpu().numpy()
    return compute_metrics(preds, horizon, day_indices, name)


print('Training utilities defined.')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N3d: GNN ABLATION EXPERIMENTS + GO/STOP GATE
# ══════════════════════════════════════════════════════════════════
t0 = time.time()
h = PARAMS['default_horizon']
hid = PARAMS['hidden_channels']
gnn_results = []

# --- A1: HGT with correlation edges only ---
print('\n=== GNN-A1: HGT (correlation only) ===')
# For this ablation, use full HGT but only include correlation edges
model_a1 = RankingHGT(781, 771, hid, METADATA,
                       num_heads=PARAMS['num_heads'],
                       num_layers=PARAMS['num_hgt_layers'],
                       dropout=PARAMS['dropout'])
# Temporarily modify build function to only use corr edges
_orig_build = build_hetero_data
def build_corr_only(day_idx, to_device=None):
    data = _orig_build(day_idx, to_device=None)
    # Zero out non-correlation edges
    data['stock', 'sector', 'stock'].edge_index = torch.zeros(2, 0, dtype=torch.long)
    data['news', 'mentions', 'stock'].edge_index = torch.zeros(2, 0, dtype=torch.long)
    data['stock', 'cooccur', 'stock'].edge_index = torch.zeros(2, 0, dtype=torch.long)
    if to_device:
        data = data.to(to_device)
    return data

build_hetero_data = build_corr_only
model_a1, hist_a1 = train_hgt(model_a1, h, train_days, val_days)
res = evaluate_hgt(model_a1, h, test_days, 'A1: HGT (corr only)')
gnn_results.append(res)
print(f'  Test IC={res["IC"]:.5f}  ICIR={res["ICIR"]:.3f}  Sharpe_LS={res["Sharpe_LS"]:.3f}')
build_hetero_data = _orig_build

# --- A2: HGT with correlation + sector ---
print('\n=== GNN-A2: HGT (corr + sector) ===')
model_a2 = RankingHGT(781, 771, hid, METADATA,
                       num_heads=PARAMS['num_heads'],
                       num_layers=PARAMS['num_hgt_layers'],
                       dropout=PARAMS['dropout'])
def build_corr_sector(day_idx, to_device=None):
    data = _orig_build(day_idx, to_device=None)
    data['news', 'mentions', 'stock'].edge_index = torch.zeros(2, 0, dtype=torch.long)
    data['stock', 'cooccur', 'stock'].edge_index = torch.zeros(2, 0, dtype=torch.long)
    if to_device:
        data = data.to(to_device)
    return data

build_hetero_data = build_corr_sector
model_a2, hist_a2 = train_hgt(model_a2, h, train_days, val_days)
res = evaluate_hgt(model_a2, h, test_days, 'A2: HGT (corr+sector)')
gnn_results.append(res)
print(f'  Test IC={res["IC"]:.5f}  ICIR={res["ICIR"]:.3f}  Sharpe_LS={res["Sharpe_LS"]:.3f}')
build_hetero_data = _orig_build

# --- A3: HGT with all 4 edge types (FULL) ---
print('\n=== GNN-A3: HGT (ALL 4 edge types) ===')
model_a3 = RankingHGT(781, 771, hid, METADATA,
                       num_heads=PARAMS['num_heads'],
                       num_layers=PARAMS['num_hgt_layers'],
                       dropout=PARAMS['dropout'])
model_a3, hist_a3 = train_hgt(model_a3, h, train_days, val_days)
res = evaluate_hgt(model_a3, h, test_days, 'A3: HGT (all 4 edges)')
gnn_results.append(res)
print(f'  Test IC={res["IC"]:.5f}  ICIR={res["ICIR"]:.3f}  Sharpe_LS={res["Sharpe_LS"]:.3f}')

# --- A4: GraphSAGE (homogeneous, corr+sector) ---
print('\n=== GNN-A4: GraphSAGE (corr+sector) ===')
model_a4 = RankingGNN(781, hid, conv_type='sage',
                       num_layers=PARAMS['num_hgt_layers'],
                       dropout=PARAMS['dropout'])
model_a4 = train_homogeneous_gnn(model_a4, h, train_days, val_days,
                                  edge_types=['corr', 'sector'])
res = evaluate_homogeneous_gnn(model_a4, h, test_days,
                                edge_types=['corr', 'sector'],
                                name='A4: SAGE (corr+sector)')
gnn_results.append(res)
print(f'  Test IC={res["IC"]:.5f}  ICIR={res["ICIR"]:.3f}  Sharpe_LS={res["Sharpe_LS"]:.3f}')

# --- A5: GAT (homogeneous, corr+sector) ---
print('\n=== GNN-A5: GAT (corr+sector) ===')
model_a5 = RankingGNN(781, hid, conv_type='gat',
                       num_layers=PARAMS['num_hgt_layers'],
                       dropout=PARAMS['dropout'])
model_a5 = train_homogeneous_gnn(model_a5, h, train_days, val_days,
                                  edge_types=['corr', 'sector'])
res = evaluate_homogeneous_gnn(model_a5, h, test_days,
                                edge_types=['corr', 'sector'],
                                name='A5: GAT (corr+sector)')
gnn_results.append(res)
print(f'  Test IC={res["IC"]:.5f}  ICIR={res["ICIR"]:.3f}  Sharpe_LS={res["Sharpe_LS"]:.3f}')

# --- Combined results ---
all_results = baseline_results + gnn_results
results_df = print_results_table(all_results, f'Full Results ({h}d horizon, test set)')

# --- Go/Stop Gate ---
best_ic = max(r['IC'] for r in all_results)
best_sharpe = max(r['Sharpe_LS'] for r in all_results)
print(f'\n=== GO/STOP GATE ===')
print(f'Best IC:        {best_ic:.5f}  (threshold: 0.03)')
print(f'Best Sharpe_LS: {best_sharpe:.3f}  (threshold: 0.5)')
if best_ic > 0.03 or best_sharpe > 0.5:
    print('>>> GO: Proceed to N4 (Horizon Ablation) + N5 (Selective Prediction)')
else:
    print('>>> STOP: All models IC ~ 0 — ranking also unpredictable')
    print('    Consider: negative result paper (EMH evidence for ranking too)')

# Save results
results_df.to_csv('experiments/v3_baseline_results.csv', index=False)
print(f'\nTotal time: {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N4: HORIZON ABLATION — 1d/5d/10d/21d/42d/63d
# Uses best GNN config from N3: GAT (corr+sector)
# ══════════════════════════════════════════════════════════════════
t0 = time.time()
hid = PARAMS['hidden_channels']

horizon_results = []

for h in PARAMS['horizons']:
    print(f'\n=== Horizon {h}d ===')

    # Train fresh GAT for this horizon (best config from N3)
    model = RankingGNN(781, hid, conv_type='gat',
                        num_layers=PARAMS['num_hgt_layers'],
                        dropout=PARAMS['dropout'])
    model = train_homogeneous_gnn(model, h, train_days, val_days,
                                   edge_types=['corr', 'sector'], verbose=True)

    # Evaluate on test set
    res = evaluate_homogeneous_gnn(model, h, test_days,
                                    edge_types=['corr', 'sector'],
                                    name=f'GAT {h}d')
    horizon_results.append(res)
    print(f'  IC={res["IC"]:.5f}  ICIR={res["ICIR"]:.3f}  '
          f'Sharpe_LS={res["Sharpe_LS"]:.3f}  Ann_LS={res["Ann_LS"]}')

    # Also evaluate LightGBM baseline for comparison
    if HAS_LGB:
        X_tr, y_tr = flatten_data(train_days, stock_features_np, labels[h], label_valid[h])
        X_va, y_va = flatten_data(val_days, stock_features_np, labels[h], label_valid[h])
        sc = StandardScaler()
        X_tr_s = sc.fit_transform(X_tr)
        X_va_s = sc.transform(X_va)

        lgb_m = lgb.LGBMRegressor(
            n_estimators=200, max_depth=5, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            random_state=SEED, n_jobs=-1, verbose=-1,
        )
        lgb_m.fit(X_tr_s, y_tr,
                  eval_set=[(X_va_s, y_va)],
                  callbacks=[lgb.early_stopping(20, verbose=False)])

        pred_lgb_h = np.zeros((num_days, num_stocks), dtype=np.float32)
        for d in range(num_days):
            pred_lgb_h[d] = lgb_m.predict(sc.transform(stock_features_np[d]))

        res_lgb = compute_metrics(pred_lgb_h, h, test_days, f'LGBM {h}d')
        horizon_results.append(res_lgb)
        print(f'  LGBM IC={res_lgb["IC"]:.5f}  Sharpe_LS={res_lgb["Sharpe_LS"]:.3f}')

    del model
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

# --- Visualization ---
horizon_df = pd.DataFrame(horizon_results)
horizon_df.to_csv('experiments/v3_horizon_ablation.csv', index=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
horizons_plot = PARAMS['horizons']

# Filter GAT and LGBM results
gat_res = [r for r in horizon_results if r['name'].startswith('GAT')]
lgb_res = [r for r in horizon_results if r['name'].startswith('LGBM')]

# IC vs Horizon
ax = axes[0]
if gat_res:
    ax.plot(horizons_plot[:len(gat_res)], [r['IC'] for r in gat_res], 'o-', label='GAT', color='tab:blue')
if lgb_res:
    ax.plot(horizons_plot[:len(lgb_res)], [r['IC'] for r in lgb_res], 's--', label='LGBM', color='tab:orange')
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.axhline(0.03, color='red', linestyle='--', alpha=0.5, label='Go threshold')
ax.set_xlabel('Prediction Horizon (trading days)')
ax.set_ylabel('Information Coefficient (IC)')
ax.set_title('IC vs Prediction Horizon')
ax.legend()
ax.grid(True, alpha=0.3)

# ICIR vs Horizon
ax = axes[1]
if gat_res:
    ax.plot(horizons_plot[:len(gat_res)], [r['ICIR'] for r in gat_res], 'o-', label='GAT', color='tab:blue')
if lgb_res:
    ax.plot(horizons_plot[:len(lgb_res)], [r['ICIR'] for r in lgb_res], 's--', label='LGBM', color='tab:orange')
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Prediction Horizon (trading days)')
ax.set_ylabel('ICIR')
ax.set_title('ICIR vs Prediction Horizon')
ax.legend()
ax.grid(True, alpha=0.3)

# Sharpe vs Horizon
ax = axes[2]
if gat_res:
    ax.plot(horizons_plot[:len(gat_res)], [r['Sharpe_LS'] for r in gat_res], 'o-', label='GAT', color='tab:blue')
if lgb_res:
    ax.plot(horizons_plot[:len(lgb_res)], [r['Sharpe_LS'] for r in lgb_res], 's--', label='LGBM', color='tab:orange')
ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='Go threshold')
ax.set_xlabel('Prediction Horizon (trading days)')
ax.set_ylabel('Long-Short Sharpe')
ax.set_title('Sharpe Ratio vs Prediction Horizon')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/v3_horizon_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

print_results_table(horizon_results, 'Horizon Ablation (test set)')
print(f'\nHorizon ablation complete in {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N5a: SELECTIVENET — 3-head GAT architecture
# Head 1: Ranking prediction (from GAT backbone)
# Head 2: Selection (learn when to trade)
# Head 3: Auxiliary (regularization)
# ══════════════════════════════════════════════════════════════════
from torch_geometric.nn import GATConv as GATConv_  # avoid re-import conflict

class SelectiveRankingGAT(nn.Module):
    """GAT + SelectiveNet for ranking with selective prediction.
    Uses homogeneous graph (stock nodes + corr+sector edges only)."""
    def __init__(self, in_channels, hidden, num_layers=2, dropout=0.3, market_dim=4):
        super().__init__()
        self.lin = nn.Linear(in_channels, hidden)
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(num_layers):
            self.convs.append(GATConv_(hidden, hidden // 4, heads=4, concat=True))
            self.norms.append(nn.LayerNorm(hidden))
        self.dropout = dropout

        self.ranking_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden // 2, 1),
        )
        self.selection_head = nn.Sequential(
            nn.Linear(hidden + market_dim, hidden // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden // 2, 1), nn.Sigmoid(),
        )
        self.auxiliary_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden // 2, 1),
        )

    def _backbone(self, x, edge_index):
        h = F.relu(self.lin(x))
        for conv, norm in zip(self.convs, self.norms):
            h_new = conv(h, edge_index)
            h = norm(F.dropout(h_new, p=self.dropout, training=self.training) + h)
        return h

    def forward(self, x, edge_index, market_context=None):
        emb = self._backbone(x, edge_index)
        ranking = self.ranking_head(emb).squeeze(-1)
        auxiliary = self.auxiliary_head(emb).squeeze(-1)
        if market_context is not None:
            mc = market_context.unsqueeze(0).expand(emb.shape[0], -1)
            sel_input = torch.cat([emb, mc], dim=-1)
        else:
            sel_input = torch.cat([emb, torch.zeros(emb.shape[0], 4, device=emb.device)], dim=-1)
        selection = self.selection_head(sel_input).squeeze(-1)
        return ranking, selection, auxiliary


def selective_loss(ranking, selection, auxiliary, target, mask, coverage_target=0.2, lam=32.0):
    """SelectiveNet loss (Geifman & El-Yaniv, ICML'19)."""
    valid_sel = selection[mask]
    valid_rank = ranking[mask]
    valid_aux = auxiliary[mask]
    valid_target = target[mask]
    if valid_sel.sum() < 1e-8:
        return torch.tensor(0.0, device=ranking.device, requires_grad=True)
    per_sample_loss = (valid_rank - valid_target) ** 2
    selective_risk = (valid_sel * per_sample_loss).sum() / (valid_sel.sum() + 1e-8)
    coverage = valid_sel.mean()
    coverage_penalty = lam * F.relu(coverage_target - coverage) ** 2
    aux_loss = F.mse_loss(valid_aux, valid_target)
    return selective_risk + coverage_penalty + aux_loss


print('SelectiveRankingGAT model defined (homogeneous, corr+sector).')
print(f'  Market context: 4-dim (VIX proxy, drawdown, realized vol, breadth)')
print(f'  Target coverage: {PARAMS["target_coverage"]*100:.0f}%')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N5b: SELECTIVENET TRAINING (GAT backbone)
# Stage 1: Train backbone + ranking + auxiliary
# Stage 2: Freeze backbone + ranking, train selection head
# ══════════════════════════════════════════════════════════════════
t0 = time.time()

# --- Helper: build homogeneous graph (corr+sector) ---
def _build_homo_graph(d, dev):
    x = stock_features_t[d].to(dev)
    snap = corr_day_to_snapshot.get(d, 0)
    ei = torch.cat([corr_snapshots[snap], sector_edge_index], dim=1).to(dev)
    return x, ei

# --- Compute market context features (T-1) ---
market_ret = returns[valid_tickers].mean(axis=1)
market_cum = (1 + market_ret).cumprod()

market_context_np = np.zeros((num_days, 4), dtype=np.float32)
for d in range(1, num_days):
    if d >= 21:
        market_context_np[d, 0] = market_ret.iloc[d-21:d].std() * np.sqrt(252)
    if d >= 63:
        peak = market_cum.iloc[d-63:d].max()
        market_context_np[d, 1] = (market_cum.iloc[d-1] - peak) / peak
    if d >= 30:
        market_context_np[d, 2] = returns.iloc[d-30:d].std(axis=0).mean()
    if d >= 5:
        breadth = (returns.iloc[d-5:d].sum(axis=0) > 0).mean()
        market_context_np[d, 3] = breadth

market_context_t = torch.tensor(market_context_np, dtype=torch.float32)
print(f'Market context: {market_context_t.shape}')

# --- Best horizon from N4 (or default) ---
best_horizon = PARAMS['default_horizon']
if 'horizon_results' in dir() and horizon_results:
    gat_only = [r for r in horizon_results if r['name'].startswith('GAT')]
    if gat_only:
        best_h = max(gat_only, key=lambda r: r['IC'])
        best_horizon = int(best_h['name'].split()[1].replace('d', ''))
        print(f'Best horizon from N4: {best_horizon}d (IC={best_h["IC"]:.5f})')

h = best_horizon
hid = PARAMS['hidden_channels']

# --- Stage 1: Train backbone + ranking + auxiliary ---
print(f'\n=== Stage 1: Train GAT backbone ({h}d horizon) ===')
sel_model = SelectiveRankingGAT(
    781, hid,
    num_layers=PARAMS['num_hgt_layers'],
    dropout=PARAMS['dropout'],
    market_dim=4,
).to(device)

optimizer = torch.optim.Adam(sel_model.parameters(), lr=PARAMS['lr'],
                              weight_decay=PARAMS['weight_decay'])

lab = labels_t[h]
valid = label_valid_t[h]
best_val = float('inf')
best_state = None
no_improve = 0

for epoch in range(PARAMS['epochs']):
    sel_model.train()
    train_loss_sum = 0
    train_count = 0
    optimizer.zero_grad()

    day_order = train_days[np.random.permutation(len(train_days))]
    for step, d in enumerate(day_order):
        x, edge_index = _build_homo_graph(d, device)
        mc = market_context_t[d].to(device)
        ranking, selection, auxiliary = sel_model(x, edge_index, mc)

        mask = valid[d].to(device)
        target = lab[d].to(device)
        if mask.sum() < 10:
            continue

        loss = F.mse_loss(ranking[mask], target[mask]) + F.mse_loss(auxiliary[mask], target[mask])
        loss = loss / PARAMS['grad_accum']
        loss.backward()
        train_loss_sum += loss.item() * PARAMS['grad_accum']
        train_count += 1

        if (step + 1) % PARAMS['grad_accum'] == 0 or step == len(day_order) - 1:
            torch.nn.utils.clip_grad_norm_(sel_model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

    avg_train = train_loss_sum / max(train_count, 1)

    # Validate
    sel_model.eval()
    val_loss_sum = 0
    val_count = 0
    with torch.no_grad():
        for d in val_days:
            x, edge_index = _build_homo_graph(d, device)
            mc = market_context_t[d].to(device)
            ranking, _, auxiliary = sel_model(x, edge_index, mc)
            mask = valid[d].to(device)
            target = lab[d].to(device)
            if mask.sum() < 10:
                continue
            loss = F.mse_loss(ranking[mask], target[mask])
            val_loss_sum += loss.item()
            val_count += 1

    avg_val = val_loss_sum / max(val_count, 1)
    if avg_val < best_val:
        best_val = avg_val
        best_state = {k: v.cpu().clone() for k, v in sel_model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1

    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1}: train={avg_train:.5f} val={avg_val:.5f}')
    if no_improve >= PARAMS['patience']:
        print(f'  Early stop at epoch {epoch+1}')
        break

sel_model.load_state_dict(best_state)
sel_model = sel_model.to(device)
print('Stage 1 complete.')

# --- Stage 2: Freeze backbone + ranking, train selection head ---
print(f'\n=== Stage 2: Train selection head ===')

for name, param in sel_model.named_parameters():
    if 'selection_head' not in name:
        param.requires_grad = False

sel_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, sel_model.parameters()),
    lr=PARAMS['lr'] * 0.1,
)

best_val_sel = float('inf')
best_state_sel = None
no_improve_sel = 0

for epoch in range(50):
    sel_model.train()
    train_loss_sum = 0
    train_count = 0
    sel_optimizer.zero_grad()

    day_order = train_days[np.random.permutation(len(train_days))]
    for step, d in enumerate(day_order):
        x, edge_index = _build_homo_graph(d, device)
        mc = market_context_t[d].to(device)
        ranking, selection, auxiliary = sel_model(x, edge_index, mc)

        mask = valid[d].to(device)
        target = lab[d].to(device)
        if mask.sum() < 10:
            continue

        loss = selective_loss(ranking, selection, auxiliary, target, mask,
                               coverage_target=PARAMS['target_coverage'],
                               lam=PARAMS['selection_lambda'])
        loss = loss / PARAMS['grad_accum']
        loss.backward()
        train_loss_sum += loss.item() * PARAMS['grad_accum']
        train_count += 1

        if (step + 1) % PARAMS['grad_accum'] == 0 or step == len(day_order) - 1:
            sel_optimizer.step()
            sel_optimizer.zero_grad()

    avg_train = train_loss_sum / max(train_count, 1)

    sel_model.eval()
    val_loss_sum = 0
    val_count = 0
    with torch.no_grad():
        for d in val_days:
            x, edge_index = _build_homo_graph(d, device)
            mc = market_context_t[d].to(device)
            ranking, selection, auxiliary = sel_model(x, edge_index, mc)
            mask = valid[d].to(device)
            target = lab[d].to(device)
            if mask.sum() < 10:
                continue
            loss = selective_loss(ranking, selection, auxiliary, target, mask,
                                   coverage_target=PARAMS['target_coverage'],
                                   lam=PARAMS['selection_lambda'])
            val_loss_sum += loss.item()
            val_count += 1

    avg_val = val_loss_sum / max(val_count, 1)
    if avg_val < best_val_sel:
        best_val_sel = avg_val
        best_state_sel = {k: v.cpu().clone() for k, v in sel_model.state_dict().items()}
        no_improve_sel = 0
    else:
        no_improve_sel += 1

    if (epoch + 1) % 10 == 0:
        coverages = []
        with torch.no_grad():
            for d in val_days[:20]:
                x, edge_index = _build_homo_graph(d, device)
                mc = market_context_t[d].to(device)
                _, sel, _ = sel_model(x, edge_index, mc)
                coverages.append(sel.mean().item())
        avg_cov = np.mean(coverages)
        print(f'  Epoch {epoch+1}: loss={avg_train:.5f} val={avg_val:.5f} coverage={avg_cov:.3f}')

    if no_improve_sel >= 15:
        print(f'  Early stop at epoch {epoch+1}')
        break

sel_model.load_state_dict(best_state_sel)
sel_model = sel_model.to(device)

for param in sel_model.parameters():
    param.requires_grad = True

print(f'\nStage 2 complete.')
print(f'SelectiveNet training total: {time.time()-t0:.1f}s')

In [ ]:
# ══════════════════════════════════════════════════════════════════
# N5c: SELECTION ANALYSIS + VISUALIZATION
# Compare: (1) Full prediction (2) Threshold (3) SelectiveNet
# ══════════════════════════════════════════════════════════════════
t0 = time.time()
h = best_horizon

# --- Collect test predictions + selection scores ---
sel_model.eval()
test_rankings = np.zeros((num_days, num_stocks), dtype=np.float32)
test_selections = np.zeros((num_days, num_stocks), dtype=np.float32)

with torch.no_grad():
    for d in test_days:
        x, edge_index = _build_homo_graph(d, device)
        mc = market_context_t[d].to(device)
        ranking, selection, _ = sel_model(x, edge_index, mc)
        test_rankings[d] = ranking.cpu().numpy()
        test_selections[d] = selection.cpu().numpy()

# --- Method 1: Full prediction (no selection) ---
res_full = compute_metrics(test_rankings, h, test_days, 'Full (100%)')

# --- Method 2: Threshold-based selection (|ranking| > percentile) ---
threshold_results = []
for cov_target in [0.05, 0.10, 0.20, 0.50, 1.0]:
    # Use ranking magnitude as confidence proxy
    preds_selective = test_rankings.copy()
    for d in test_days:
        scores = np.abs(test_rankings[d])
        if cov_target < 1.0:
            threshold = np.percentile(scores, (1 - cov_target) * 100)
            mask = scores < threshold
            preds_selective[d][mask] = 0  # zero out low-confidence
    res = compute_metrics(preds_selective, h, test_days, f'Threshold @{cov_target*100:.0f}%')
    threshold_results.append(res)

# --- Method 3: SelectiveNet-based selection ---
selectivenet_results = []
for cov_target in [0.05, 0.10, 0.20, 0.50, 1.0]:
    preds_sel = test_rankings.copy()
    for d in test_days:
        sel_scores = test_selections[d]
        if cov_target < 1.0:
            threshold = np.percentile(sel_scores, (1 - cov_target) * 100)
            mask = sel_scores < threshold
            preds_sel[d][mask] = 0
    res = compute_metrics(preds_sel, h, test_days, f'SelectiveNet @{cov_target*100:.0f}%')
    selectivenet_results.append(res)

# --- Combined table ---
all_sel_results = [res_full] + threshold_results + selectivenet_results
print_results_table(all_sel_results, f'Selective Prediction Comparison ({h}d horizon)')

# --- Visualization ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

coverages = [5, 10, 20, 50, 100]

# Plot 1: IC vs Coverage
ax = axes[0, 0]
thr_ics = [r['IC'] for r in threshold_results]
sel_ics = [r['IC'] for r in selectivenet_results]
ax.plot(coverages, thr_ics, 'o-', label='Threshold', color='tab:orange')
ax.plot(coverages, sel_ics, 's-', label='SelectiveNet', color='tab:blue')
ax.axhline(res_full['IC'], color='gray', linestyle=':', label='Full')
ax.set_xlabel('Coverage (%)')
ax.set_ylabel('IC')
ax.set_title('IC vs Coverage')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Sharpe vs Coverage
ax = axes[0, 1]
thr_sh = [r['Sharpe_LS'] for r in threshold_results]
sel_sh = [r['Sharpe_LS'] for r in selectivenet_results]
ax.plot(coverages, thr_sh, 'o-', label='Threshold', color='tab:orange')
ax.plot(coverages, sel_sh, 's-', label='SelectiveNet', color='tab:blue')
ax.axhline(res_full['Sharpe_LS'], color='gray', linestyle=':', label='Full')
ax.set_xlabel('Coverage (%)')
ax.set_ylabel('Long-Short Sharpe')
ax.set_title('Sharpe vs Coverage')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Selection score distribution
ax = axes[0, 2]
all_sel_scores = test_selections[test_days].flatten()
ax.hist(all_sel_scores, bins=50, alpha=0.7, color='tab:blue', density=True)
ax.set_xlabel('Selection Score')
ax.set_ylabel('Density')
ax.set_title('SelectiveNet Score Distribution')
ax.grid(True, alpha=0.3)

# Plot 4: Per-month coverage + market vol overlay
ax = axes[1, 0]
monthly_coverage = []
monthly_vol = []
monthly_dates = []
for d in test_days:
    monthly_dates.append(all_dates[d])
    monthly_coverage.append(test_selections[d].mean())
    monthly_vol.append(market_context_np[d, 0])  # market vol

ax.plot(monthly_dates, monthly_coverage, alpha=0.3, color='tab:blue', label='Daily coverage')
# Rolling average
if len(monthly_coverage) > 21:
    rolling_cov = pd.Series(monthly_coverage).rolling(21).mean()
    ax.plot(monthly_dates, rolling_cov, color='tab:blue', linewidth=2, label='21d avg coverage')
ax2 = ax.twinx()
ax2.plot(monthly_dates, monthly_vol, alpha=0.5, color='tab:red', label='Market vol')
ax.set_xlabel('Date')
ax.set_ylabel('SelectiveNet Coverage', color='tab:blue')
ax2.set_ylabel('Market Vol (annualized)', color='tab:red')
ax.set_title('Coverage vs Market Volatility')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')

# Plot 5: Jaccard overlap between threshold and SelectiveNet
ax = axes[1, 1]
jaccard_by_cov = []
for cov_target in [0.05, 0.10, 0.20, 0.50]:
    jaccards = []
    for d in test_days[:100]:  # sample for speed
        # Threshold selection
        thr_scores = np.abs(test_rankings[d])
        thr_thresh = np.percentile(thr_scores, (1 - cov_target) * 100)
        thr_selected = set(np.where(thr_scores >= thr_thresh)[0])

        # SelectiveNet selection
        sel_scores = test_selections[d]
        sel_thresh = np.percentile(sel_scores, (1 - cov_target) * 100)
        sel_selected = set(np.where(sel_scores >= sel_thresh)[0])

        # Jaccard
        if len(thr_selected | sel_selected) > 0:
            j = len(thr_selected & sel_selected) / len(thr_selected | sel_selected)
            jaccards.append(j)
    jaccard_by_cov.append(np.mean(jaccards) if jaccards else 0)

ax.bar([5, 10, 20, 50], jaccard_by_cov, color='tab:green', alpha=0.7)
ax.set_xlabel('Coverage (%)')
ax.set_ylabel('Jaccard Similarity')
ax.set_title('Threshold vs SelectiveNet Overlap')
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# Plot 6: Cumulative L/S return (full vs selective)
ax = axes[1, 2]
fwd_raw = prices.pct_change(h).shift(-h).values
for label_name, preds in [('Full', test_rankings), ('SelectiveNet @20%', preds_sel)]:
    cum_ret = []
    running = 0
    for d in test_days:
        mask_v = label_valid[h][d]
        if mask_v.sum() < 2 * PARAMS['top_k']:
            continue
        p = preds[d].copy()
        p[~mask_v] = np.nan
        valid_idx = np.where(mask_v)[0]
        scores = p[valid_idx]
        ranks = np.argsort(np.argsort(-scores))
        top = valid_idx[ranks < PARAMS['top_k']]
        bot = valid_idx[ranks >= len(scores) - PARAMS['top_k']]
        ret = np.nanmean(fwd_raw[d][top]) - np.nanmean(fwd_raw[d][bot])
        running += ret
        cum_ret.append(running)
    ax.plot(range(len(cum_ret)), cum_ret, label=label_name, alpha=0.8)

ax.set_xlabel('Rebalance Period')
ax.set_ylabel('Cumulative L/S Return')
ax.set_title('Cumulative Long-Short Return')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/v3_selective_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nSelective prediction analysis complete in {time.time()-t0:.1f}s')

## Observations & Next Steps

### Key Results

| Metric | Target | Actual | Status |
|--------|--------|--------|--------|
| Best IC (any horizon) | > 0.03 | TBD | |
| HGT IC > LightGBM IC | HGT wins | TBD | |
| Selective IC@20% > Full IC | Selective wins | TBD | |
| Long-short Sharpe > 0.5 | Economically significant | TBD | |
| Horizon pattern | Clear trend | TBD | |

### Observations (fill after running)

1. **Best horizon**: ___ d (IC = ___)
2. **Graph contribution**: HGT vs LightGBM delta = ___
3. **Edge type importance**: ___
4. **SelectiveNet coverage**: Average ___, varies with market vol? ___
5. **IC spectrum**: Short-term (1d) IC = ___, Long-term (63d) IC = ___

### Next Steps

- [ ] If Go: Walk-forward validation (N6a)
- [ ] If Go: Attention weight analysis (which edge types matter)
- [ ] If Go: Paper figures generation
- [ ] If Stop: Frame as comprehensive negative result paper